### Data Ingestion

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source.txt", "pages": 1,
        "author": "John Doe", 
        "date_created": "2024-06-01"
    },
)
doc

Document(metadata={'source': 'test_source.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is a test document.')

In [3]:
## Create a simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts = {
    # Path to Python language documentation file
    "../data/text_files/python.txt": """Python is a high-level, interpreted programming language.
- Easy to learn and read with clean syntax
- Versatile: web development, data science, AI, automation
- Extensive standard library and third-party packages
- Dynamic typing with strong type inference
- Cross-platform support (Windows, Linux, macOS)
- Large community and excellent documentation
- Supports multiple programming paradigms""",
    # Path to C language documentation file
    "../data/text_files/c.txt": """C is a low-level, compiled programming language.
- Provides direct memory access and system-level control
- Foundation for many modern languages (C++, Java, Python)
- Highly efficient and fast execution
- Requires manual memory management
- Portable across different hardware platforms
- Used in operating systems, embedded systems, drivers
- Rich set of operators and control structures""",
}

In [5]:
for filepath, content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [6]:
# TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python.txt", encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]


In [7]:
# DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    "../data/text_files", 
    glob="*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={"encoding": "utf-8"}, 
    show_progress=False
)
documents = loader.load()
print(documents)    

[Document(metadata={'source': '../data/text_files/c.txt'}, page_content='C is a low-level, compiled programming language.\n- Provides direct memory access and system-level control\n- Foundation for many modern languages (C++, Java, Python)\n- Highly efficient and fast execution\n- Requires manual memory management\n- Portable across different hardware platforms\n- Used in operating systems, embedded systems, drivers\n- Rich set of operators and control structures'), Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]


### Embedding

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded.")
                             
        print(f"Generating embeddings for {len(text)} texts...")
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings

In [10]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [11]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection of PDF document embeddings"})

            print(f"Vector store initialized successfully. Collection name: '{self.collection_name}' Existing document at collection '{self.collection.count()}'")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise